# ⚙️ Notebook 3: Feature Engineering
**Project:** Savanna Bank Kenya — Customer Churn Analysis  
**Author:** Okabinini Harriet-Data Analytics Team  
**Date:** 2026-05-08

---

## Objectives
- Validate and extend the engineered features from the cleaning notebook
- Analyse the distribution and predictive signal of each derived feature
- Prepare a final feature matrix for modelling

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

CLEAN_PATH = "../.data/cleaned/bank_customer_churn_clean.csv"
VISUALS    = "../outputs/visuals/"

df = pd.read_csv(CLEAN_PATH)
df["account_open_date"]     = pd.to_datetime(df["account_open_date"])
df["last_transaction_date"] = pd.to_datetime(df["last_transaction_date"])
today = pd.Timestamp("2026-05-08")
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

## 2. Engineered Feature Inventory

| Feature | Formula | Business Rationale |
|---------|---------|--------------------|
| `days_since_last_txn` | today - last_transaction_date | Direct inactivity signal |
| `account_age_months` | months between open date and today | Longer tenure = lower churn risk |
| `has_loan` | 1 if loan_type != "No Loan" | Loan customers have stronger ties |
| `loan_to_income_ratio` | loan_amount / monthly_income | Debt burden relative to income |
| `repayment_burden` | monthly_repayment / monthly_income | Monthly cash pressure indicator |
| `balance_to_income_ratio` | balance / monthly_income | Savings buffer — protection against churn |

## 3. Feature Distributions

In [ ]:
eng_features = [
    "days_since_last_txn", "account_age_months",
    "loan_to_income_ratio", "repayment_burden", "balance_to_income_ratio"
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
churners = df[df["churn_label"] == 1]
active   = df[df["churn_label"] == 0]

for idx, col in enumerate(eng_features):
    ax = axes[idx]
    plot_data = df[col].clip(
        upper=df[col].quantile(0.99)
    )
    ax.hist(active[col].clip(upper=df[col].quantile(0.99)), bins=40,
            alpha=0.6, color="#1D9E75", label="Active", density=True, edgecolor="white")
    ax.hist(churners[col].clip(upper=df[col].quantile(0.99)), bins=40,
            alpha=0.6, color="#E24B4A", label="Churned", density=True, edgecolor="white")
    ax.set_title(col.replace("_", " ").title(), fontweight="bold")
    ax.legend(fontsize=9)

axes[5].axis("off")
plt.suptitle("Engineered Feature Distributions by Churn Status", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(VISUALS + "09_engineered_feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Risk Segment: Repayment Burden Bands

In [ ]:
# Bin repayment_burden into risk tiers
bins   = [-0.001, 0.2, 0.4, 0.6, 1.0, np.inf]
labels = ["Very Low (<20%)", "Low (20-40%)", "Medium (40-60%)", "High (60-100%)", "Critical (>100%)"]
df["repayment_band"] = pd.cut(df["repayment_burden"], bins=bins, labels=labels)

band_churn = df.groupby("repayment_band", observed=True)["churn_label"].agg(["count","mean"])
band_churn.columns = ["customers", "churn_rate"]
band_churn["churn_rate_pct"] = (band_churn["churn_rate"] * 100).round(1)
print(band_churn)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#1D9E75","#9FE1CB","#FAC775","#E24B4A","#791F1F"]
bars = ax.bar(band_churn.index, band_churn["churn_rate_pct"], color=colors, edgecolor="white")
ax.set_title("Churn Rate by Repayment Burden Band", fontweight="bold")
ax.set_ylabel("Churn Rate (%)")
ax.set_xlabel("Repayment Burden Tier")
ax.axhline(df["churn_label"].mean()*100, color="navy", linestyle="--", label="Overall avg")
ax.legend()
for bar, val in zip(bars, band_churn["churn_rate_pct"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f"{val}%", ha="center", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig(VISUALS + "10_repayment_burden_churn.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Feature Correlation with Churn (Point-Biserial)

In [ ]:
from scipy.stats import pointbiserialr

num_cols = [
    "account_balance_ksh","monthly_income_ksh","credit_score",
    "num_transactions_30d","num_products_held","days_since_last_txn",
    "account_age_months","repayment_burden","loan_to_income_ratio","balance_to_income_ratio"
]

results = []
for col in num_cols:
    r, p = pointbiserialr(df["churn_label"], df[col].fillna(0))
    results.append({"feature": col, "correlation": round(r, 4), "p_value": round(p, 4)})

corr_df = pd.DataFrame(results).sort_values("correlation", key=abs, ascending=False)
print(corr_df.to_string(index=False))

## 6. Final Feature Matrix Summary

In [ ]:
feature_cols = [
    "gender","region","marital_status","employment_status","account_type",
    "months_active","account_balance_ksh","monthly_income_ksh","credit_score",
    "num_transactions_30d","num_products_held","loan_type",
    "days_since_last_txn","account_age_months","loan_to_income_ratio",
    "repayment_burden","balance_to_income_ratio","has_loan"
]

print(f"Total features for modelling: {len(feature_cols)}")
print(f"  - Categorical: {sum(df[f].dtype == object for f in feature_cols)}")
print(f"  - Numeric:     {sum(df[f].dtype != object for f in feature_cols)}")
print(f"\nClass balance:")
print(df["churn_label"].value_counts())
print(f"\nImbalance ratio: 1:{df['churn_label'].value_counts()[0]//df['churn_label'].value_counts()[1]}")